# RAG system - example usage

In [119]:
import os

from dotenv import load_dotenv
load_dotenv("./.env")

from google import genai

import chromadb

from IPython.display import Markdown, display

## Embedding the query

In [120]:
query = "Bardzo boli mnie głowa i potrzebuje szybko zabić ból, mam uczulenie na salicyl, więc tak żebym się nie przekręcił"

In [121]:
from google.genai.models import types
vector = None

try:
    client = genai.Client(
        api_key=os.environ["GOOGLE_API_KEY"]
    )

    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[query],
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY"
        )
    )

    vector = response.embeddings[0].values

except Exception as e:
    print(e)

## Database quering

In [122]:
client_chroma = chromadb.PersistentClient("./chroma_db")
collection =client_chroma.get_collection("documents")

In [123]:
results = collection.query(
    query_embeddings=[vector],
    n_results=3
)
documents_rag = [doc for doc in results['documents'] ]
print(results["metadatas"])

[[{'med_name': 'Alka-Seltzer®'}, {'med_name': 'Acatar Zatoki'}, {'med_name': 'Almozen'}]]


In [124]:
def rag_query(query, context):
    prompt = f"""
        Jesteś asystentem medycznym działającym w systemie RAG.

        ZASADY PODSTAWOWE:
        - Odpowiadasz WYŁĄCZNIE na podstawie dostarczonego kontekstu.
        - Jeśli w kontekście nie ma informacji potrzebnych do odpowiedzi, napisz: "brak danych".
        - Nie używasz wiedzy spoza kontekstu.
        - Nie informujesz użytkownika o istnieniu kontekstu ani systemu RAG.

        ZASADY DOTYCZĄCE LEKÓW:
        - Jeśli w kontekście znajdują się leki, możesz je opisać i porównać.
        - Nie pomijasz leków tylko dlatego, że są mniej odpowiednie - przedstawiasz je obiektywnie.
        - Możesz wskazać, które opcje wydają się bardziej adekwatne w kontekście objawów, jeśli wynika to bezpośrednio z danych w kontekście.

        DAWKOWANIE I BEZPIECZEŃSTWO:
        - Informacje o dawkowaniu, przeciwwskazaniach i działaniu podawaj WYŁĄCZNIE jeśli są obecne w kontekście (np. w ulotkach).
        - Nie tworzysz żadnych nowych dawek ani zaleceń.

        STRUKTURA ODPOWIEDZI:
        - Najpierw krótka analiza objawów (jeśli możliwa z kontekstu)
        - Następnie lista możliwych leków z kontekstu
        - Przy każdym leku: działanie, wskazania, przeciwwskazania (jeśli są w kontekście)
        - Na końcu krótkie podsumowanie opcji

        TON:
        - jasny, medyczny, neutralny
        - bez emocjonalnych sformułowań

        KONTEKST:
        {context}

        PYTANIE:
        {query}

        ODPOWIEDŹ:
    """

    client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
    response = client.models.generate_content(
        model = "gemini-2.5-flash",
        contents = prompt,
    )

    return response.text

In [125]:
context = "\n\n".join(f"[DOC {i}] {doc}" for i, doc in enumerate(documents_rag))
model_response = rag_query(query, context)

In [126]:
display(Markdown(model_response))

Analiza objawów wskazuje na silny ból głowy oraz kluczowe przeciwwskazanie w postaci uczulenia na salicylany. Konieczne jest wyeliminowanie leków zawierających tę substancję lub pokrewne.

Na podstawie dostarczonego kontekstu, dostępne są następujące leki, które można rozważyć w kontekście bólu głowy:

1.  **Alka-Seltzer®**
    *   **Działanie:** Kwas acetylosalicylowy działa przeciwbólowo, przeciwzapalnie i przeciwgorączkowo. Cytrynian sodu ma właściwości buforujące, zobojętniając nadmiar kwasu solnego w żołądku.
    *   **Wskazania:** Dolegliwości bólowe o lekkim i średnim nasileniu, np.: bóle głowy, bóle mięśniowe, bóle zębów. Ból i gorączka w przebiegu przeziębienia i grypy.
    *   **Przeciwwskazania:** Leku Alka-Seltzer® **nie wolno stosować**, jeśli pacjent ma uczulenie na substancję czynną - kwas acetylosalicylowy, inne salicylany lub którykolwiek z pozostałych składników leku. Wskazane uczulenie na salicyl stanowi bezwzględne przeciwwskazanie.

2.  **Acatar Zatoki**
    *   **Działanie:** Acatar Zatoki zawiera ibuprofen (niesteroidowy lek przeciwzapalny - NLPZ, działający przeciwbólowo, przeciwzapalnie i przeciwgorączkowo) oraz pseudoefedryny chlorowodorek (zmniejsza obrzęk błony śluzowej nosa).
    *   **Wskazania:** Doraźne łagodzenie objawów grypy i przeziębienia, takich jak: ból i niedrożność zatok obocznych nosa, katar, ból głowy, gorączka, bóle stawowo-mięśniowe.
    *   **Przeciwwskazania:** Leku Acatar Zatoki **nie wolno stosować**, jeśli pacjent ma uczulenie na substancje czynne lub inne niesteroidowe leki przeciwzapalne (NLPZ). Jest również przeciwwskazany u pacjentów, u których po przyjęciu kwasu acetylosalicylowego lub innych niesteroidowych leków przeciwzapalnych występowały kiedykolwiek w przeszłości objawy alergii. Ze względu na zgłoszone uczulenie na salicyl (kwas acetylosalicylowy) oraz fakt, że ibuprofen należy do grupy NLPZ, lek ten jest przeciwwskazany.

3.  **Almozen**
    *   **Działanie:** Almozen jest lekiem przeciwmigrenowym, należącym do selektywnych agonistów receptora serotoninowego. Wiąże się z receptorami serotoninowymi w naczyniach krwionośnych w mózgu, powodując ich zwężenie i zmniejszając reakcję zapalną związaną z migreną.
    *   **Wskazania:** Łagodzenie bólów głowy związanych z napadami migreny z aurą lub bez aury.
    *   **Przeciwwskazania:** Uczulenie na almotryptan lub którykolwiek z pozostałych składników leku. Inne przeciwwskazania obejmują choroby ograniczające dopływ krwi do serca (np. zawał serca, ból w klatce piersiowej), ciężkie lub niekontrolowane nadciśnienie tętnicze, udar mózgu, niedrożność naczyń krwionośnych w kończynach, jednoczesne przyjmowanie innych leków przeciwmigrenowych z grupy ergotaminy lub innych agonistów serotoniny, oraz ciężka choroba wątroby. Brak przeciwwskazań związanych z uczuleniem na salicylany lub inne NLPZ.
    *   **Dawkowanie (dorośli w wieku od 18 do 65 lat):** Zalecana dawka to jedna tabletka 12,5 mg, którą należy przyjąć najwcześniej jak to możliwe po wystąpieniu napadu migreny. Jeżeli napad migreny nie ustąpi, nie należy przyjmować więcej niż jednej tabletki podczas tego samego napadu. Jeśli wystąpi kolejny napad migreny w ciągu 24 godzin, można przyjąć drugą tabletkę w dawce 12,5 mg, zachowując przynajmniej 2-godzinną przerwę między dawkami. Maksymalna dawka dobowa to dwie tabletki (12,5 mg). Tabletkę należy połknąć, popijając płynem, z posiłkiem lub niezależnie od posiłków.

**Podsumowanie opcji:**
Ze względu na zgłoszone uczulenie na salicyl (kwas acetylosalicylowy), leki **Alka-Seltzer®** oraz **Acatar Zatoki** są przeciwwskazane.

Jedynym lekiem z dostarczonego kontekstu, który nie jest przeciwwskazany w przypadku uczulenia na salicylany i może łagodzić silny ból głowy (jeśli jest on napadem migreny), jest **Almozen**. Należy jednak pamiętać, że Almozen jest wskazany specjalnie w leczeniu napadów migreny.